# Automation Displacement Score — 830 BLS Occupations

This notebook runs a four-parameter model to score every US occupation in the [BLS SOC 2018](https://www.bls.gov/soc/) system on its risk of displacement by **robots** and **AI**, then lets you explore the results interactively.

## The formula

```
Displacement = (RobotTech × PhysShare + AItech × CogShare) × (1 − Barrier)
```

| Parameter | What it measures | Source |
|-----------|-----------------|--------|
| **RobotTech** | Technical capability of robots for the physical tasks of this job | IFR 2022 industry robot density + O*NET physical task structure |
| **AItech** | LLM capability for the cognitive tasks | Eloundou et al. (2023) β-scores + Anthropic Economic Index (2026) |
| **PhysShare** | What fraction of the job's economic value is physical vs cognitive | O*NET Work Activities elements 4.A.3.* |
| **Barrier** | Social, legal & institutional friction preventing displacement even if technically feasible | max(constitutional bar, licensing+liability, consumer preference via Pew surveys) |

The `max()` rule for Barrier: one absolute barrier (e.g., constitutional requirement for elected officials) is enough to suppress displacement regardless of technical capability.

---
Built on methodology from [Anthropic Labor Market Impacts (2026)](https://www.anthropic.com/research/labor-market-impacts).

In [ ]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn -q

In [ ]:
# If running from the repo, import the scoring engine directly
# Otherwise, we'll load the pre-scored CSV

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## Option A — Load pre-scored results (fast)
Use this if you just want to explore the data.

In [ ]:
# Load pre-scored CSV (download from GitHub or upload manually)
df = pd.read_csv('bls_automation_scores.csv')
print(f'Loaded {len(df)} occupations')
df.head(10)

## Option B — Re-run the full scoring pipeline
This re-derives all scores from the algorithm. Takes ~5 seconds.

In [ ]:
# Only run this cell if you want to regenerate scores from scratch
# Requires score_engine.py, occupations.py, and run.py in the same directory

import sys, importlib
sys.path.insert(0, '.')

from score_engine import score_occupation
from occupations import ALL_OCCUPATIONS

records = [score_occupation(soc, title, grp) for soc, title, grp in ALL_OCCUPATIONS]
df = pd.DataFrame(records)
print(f'Scored {len(df)} occupations')
df.head()

## 1. Summary statistics

In [ ]:
print('=== Displacement Score Distribution ===')
bins = [
    ('Very high (65%+)',  df['displacement'] >= 65),
    ('High (45–65%)',    (df['displacement'] >= 45) & (df['displacement'] < 65)),
    ('Moderate (25–45%)',(df['displacement'] >= 25) & (df['displacement'] < 45)),
    ('Low (10–25%)',     (df['displacement'] >= 10) & (df['displacement'] < 25)),
    ('Resistant (<10%)', df['displacement'] < 10),
]
for label, mask in bins:
    n = mask.sum()
    print(f'  {label:22s}: {n:4d} occupations  ({n/len(df)*100:.1f}%)')

print(f'\nMean displacement: {df.displacement.mean():.1f}%')
print(f'Median displacement: {df.displacement.median():.1f}%')
print(f'\nPrimary threat breakdown:')
print(df.primary_threat.value_counts().to_string())

## 2. Top and bottom occupations

In [ ]:
cols = ['soc','title','displacement','primary_threat','ai_tech','robot_tech','phys_share','barrier']

print('=== 20 Most Displaced Occupations ===')
display(df.nlargest(20, 'displacement')[cols])

print('\n=== 20 Most Resistant Occupations ===')
display(df.nsmallest(20, 'displacement')[cols])

## 3. Group averages

In [ ]:
grp_avg = df.groupby('group_name')['displacement'].agg(['mean','count']).round(1)
grp_avg.columns = ['avg_displacement', 'n_occupations']
grp_avg = grp_avg.sort_values('avg_displacement', ascending=False)
print(grp_avg.to_string())

## 4. Visualisation — group bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

g = grp_avg.sort_values('avg_displacement')

def bar_color(v):
    if v >= 65: return '#E24B4A'
    if v >= 45: return '#D85A30'
    if v >= 25: return '#EF9F27'
    if v >= 10: return '#5DCAA5'
    return '#1D9E75'

colors = [bar_color(v) for v in g['avg_displacement']]
bars = ax.barh(g.index, g['avg_displacement'], color=colors, height=0.7)

for bar, val in zip(bars, g['avg_displacement']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=9, color='#444')

ax.set_xlabel('Average displacement risk (%)', fontsize=10)
ax.set_xlim(0, 85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=9)
ax.set_title('Average automation displacement risk by BLS major occupation group', fontsize=11, pad=12)

patches = [
    mpatches.Patch(color='#E24B4A', label='Very high (65%+)'),
    mpatches.Patch(color='#D85A30', label='High (45–65%)'),
    mpatches.Patch(color='#EF9F27', label='Moderate (25–45%)'),
    mpatches.Patch(color='#5DCAA5', label='Low (10–25%)'),
    mpatches.Patch(color='#1D9E75', label='Resistant (<10%)'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('group_averages.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scatter plot — AI tech vs Robot tech

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colors_scatter = df['displacement'].apply(bar_color)

scatter = ax.scatter(
    df['ai_tech'], df['robot_tech'],
    c=colors_scatter, alpha=0.6, s=20, linewidths=0
)

# Annotate a few key occupations
labels_to_show = {
    '15-1251': 'Computer Programmers',
    '43-8011': 'Data Entry Keyers',
    '51-4122': 'Welding Machine Ops',
    '51-9111': 'Packaging Machine Ops',
    '21-2011': 'Clergy',
    '31-9011': 'Massage Therapists',
    '29-1066': 'Psychiatrists',
    '11-1031': 'Legislators',
}
for soc, label in labels_to_show.items():
    row = df[df['soc']==soc]
    if not row.empty:
        ax.annotate(label, (row['ai_tech'].iloc[0], row['robot_tech'].iloc[0]),
                   fontsize=7.5, xytext=(5,3), textcoords='offset points',
                   color='#333')

ax.set_xlabel('AItech score (Eloundou β-calibrated)', fontsize=10)
ax.set_ylabel('RobotTech score (IFR-calibrated)', fontsize=10)
ax.set_title('AI vs Robot technical capability — 830 BLS occupations\n(color = displacement risk after applying barriers)', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

patches = [
    mpatches.Patch(color='#E24B4A', label='Very high (65%+)'),
    mpatches.Patch(color='#D85A30', label='High (45–65%)'),
    mpatches.Patch(color='#EF9F27', label='Moderate (25–45%)'),
    mpatches.Patch(color='#5DCAA5', label='Low (10–25%)'),
    mpatches.Patch(color='#1D9E75', label='Resistant (<10%)'),
]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.savefig('ai_vs_robot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Search — look up any occupation

In [ ]:
# Change search_term to look up any occupation
search_term = 'nurse'

results = df[df['title'].str.lower().str.contains(search_term.lower())]
print(f'Found {len(results)} occupations matching "{search_term}":\n')
display(results[cols].sort_values('displacement', ascending=False))

## 7. Breakdown for a specific occupation

See the exact arithmetic behind any score.

In [ ]:
def breakdown(soc_code):
    row = df[df['soc'] == soc_code]
    if row.empty:
        print(f'SOC {soc_code} not found')
        return
    r = row.iloc[0]
    cog = 100 - r.phys_share
    robot_contrib = round(r.robot_tech * r.phys_share / 100, 1)
    ai_contrib    = round(r.ai_tech    * cog          / 100, 1)
    raw           = round(robot_contrib + ai_contrib, 1)

    print(f"""{'─'*55}
{r['title']} ({r.soc})
Group: {r.group_name}
{'─'*55}
Parameters:
  RobotTech  = {r.robot_tech:3}   (IFR industry density)
  AItech     = {r.ai_tech:3}   (Eloundou β-score calibrated)
  PhysShare  = {r.phys_share:3}%  (O*NET work activities ratio)
  Barrier    = {r.barrier:3}%  (max of legal/social/licensing bars)

Step 1 — Raw capability:
  Robot contribution = {r.robot_tech} × {r.phys_share}/100 = {robot_contrib}
  AI contribution    = {r.ai_tech} × {cog}/100     = {ai_contrib}
  Raw                = {robot_contrib} + {ai_contrib}         = {raw}

Step 2 — Apply barrier:
  Displacement = {raw} × (1 − {r.barrier}/100)
               = {raw} × {round(1-r.barrier/100, 2)}
               = {r.displacement}%

Primary threat: {r.primary_threat}
{'─'*55}""")

# Try different SOC codes
breakdown('43-8011')   # Data Entry Keyers
breakdown('11-1031')   # Legislators
breakdown('51-4122')   # Welding Machine Operators

## 8. Export filtered results

In [ ]:
# Export any filtered slice to CSV

# Example: all occupations in the Healthcare Practitioners group
subset = df[df['group_name'] == 'Healthcare Practitioners'].sort_values('displacement', ascending=False)
subset.to_csv('healthcare_displacement.csv', index=False)
print(f'Exported {len(subset)} rows to healthcare_displacement.csv')

# Example: all very high risk occupations
very_high = df[df['displacement'] >= 65].sort_values('displacement', ascending=False)
very_high.to_csv('very_high_risk.csv', index=False)
print(f'Exported {len(very_high)} rows to very_high_risk.csv')

## Going fully empirical

The current pipeline uses group-level baselines derived from published summaries of O*NET, IFR, and Eloundou data. To replace these with live queries:

### O*NET PhysShare (free API, requires registration at [onetcenter.org](https://services.onetcenter.org/developer/home))

```python
import requests

def get_phys_share(soc_code, api_key):
    url = f'https://services.onetcenter.org/ws/online/occupations/{soc_code}/details/work_activities'
    r = requests.get(url, headers={'Authorization': f'Basic {api_key}'})
    data = r.json()
    
    physical_ids = ['4.A.3.a.1','4.A.3.a.2','4.A.3.a.3','4.A.3.a.4',
                    '4.A.3.b.1','4.A.3.b.2','4.A.3.b.3','4.A.3.b.4']
    
    elements = data.get('element', [])
    phys = sum(e['score']['value'] for e in elements if e['id'] in physical_ids)
    total = sum(e['score']['value'] for e in elements)
    return phys / total if total > 0 else 0.1
```

### Eloundou β scores (public data)

Download `gptsRgpts_occ_lvl.csv` from:
```
https://github.com/EIG-Research/AI-unemployment/blob/main/data/gptsRgpts_occ_lvl.csv
```
Then merge on SOC code to get published β averages.

### IFR robot density

Available via IFR annual subscription. Use the industry density table in `score_engine.py` as a free approximation (derived from published IFR summaries).

---

## References

- Eloundou et al. (2023). GPTs are GPTs. *Science*, 384, 1306–1308. https://arxiv.org/abs/2303.10130
- Massenkoff & McCrory (2026). Labor market impacts of AI. Anthropic Economic Index. https://www.anthropic.com/research/labor-market-impacts
- Acemoglu & Restrepo (2020). Robots and Jobs. *Journal of Political Economy*.
- IFR (2022). World Robotics Report. https://ifr.org
- Pew Research Center (2014). AI, Robotics, and the Future of Jobs.